In [2]:
import duckdb
import pandas as pd
import os

from datetime import datetime


In [3]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [7]:
df = pd.read_csv('../landing/z0019_1.csv', sep=';')

In [8]:
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10003,PREGO,BT10,100,50


In [17]:
arquivo = 'z0019_2.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10004,SERRA,BT50,100,200,z0019_2.csv,2026-07-22 19:34:44.059165
1,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-07-22 19:34:44.059165
2,10003,PREGO,BT10,100,60,z0019_2.csv,2026-07-22 19:34:44.059165


In [12]:
con.execute(""" 
    CREATE TABLE IF NOT EXISTS bronze_produtos (
        NATBR VARCHAR,
        MAKTX VARCHAR,
        WERKS VARCHAR,
        MAINS VARCHAR,
        LABST VARCHAR,
        nome_arquivo VARCHAR,
        data_ingestao TIMESTAMP
        )
 """)

In [18]:
con.execute("INSERT INTO bronze_produtos SELECT * FROM df")

In [21]:
resultado = con.execute(""" SELECT * FROM bronze_z0019""").fetchdf()
print(resultado)

   NATBR     MAKTX WERKS MAINS LABST nome_arquivo              data_ingestao
0  10001  PARAFUSO  BT10   100   100  z0019_1.csv 2026-07-22 19:29:38.019500
1  10002   MARTELO  BT50   100  1500  z0019_1.csv 2026-07-22 19:29:38.019500
2  10003     PREGO  BT10   100    50  z0019_1.csv 2026-07-22 19:29:38.019500
3  10004     SERRA  BT50   100   200  z0019_2.csv 2026-07-22 19:34:44.059165
4  10005   MACHADO  BT50   100   100  z0019_2.csv 2026-07-22 19:34:44.059165
5  10003     PREGO  BT10   100    60  z0019_2.csv 2026-07-22 19:34:44.059165


In [20]:
con.execute("ALTER TABLE bronze_produtos RENAME TO bronze_z0019")

In [22]:
con.close()